<a href="https://colab.research.google.com/github/FernandaChacara/pml_final_project/blob/main/notebooks/ivv_vineyard_area.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clean IVV Vineyard Area Data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FernandaChacara/pml_final_project/blob/main/notebooks/ivv_vineyard_area.ipynb)

This notebook cleans the IVV vineyard area file and saves a processed CSV that can be joined to the wine production dataset as an explanatory variable.

## 1. Install Packages

In [10]:
!pip install -q pandas numpy openpyxl xlrd lxml scikit-learn

## 2. Clone or Update Repository

In [11]:
from pathlib import Path
from datetime import datetime
import re
import shutil
import subprocess
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

REPO_URL = "https://github.com/FernandaChacara/pml_final_project.git"
ROOT_DIR = Path("/content/pml_final_project")

if ROOT_DIR.exists():
    print("Repository already exists. Pulling latest changes...")
    %cd /content/pml_final_project
    !git pull
else:
    !git clone {REPO_URL} /content/pml_final_project
    %cd /content/pml_final_project

RAW_XLS = ROOT_DIR / "data" / "raw" / "Area_total_de_vinha_PT_2025.xls"
RAW_XLSX = ROOT_DIR / "data" / "raw" / "Area_total_de_vinha_PT_2025.xlsx"
OUTPUT_FILE = ROOT_DIR / "data" / "processed" / "vineyard_area_by_region_clean.csv"

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

print("Root directory:", ROOT_DIR)
print("Raw XLS file:", RAW_XLS)
print("Raw XLSX file:", RAW_XLSX)
print("Output file:", OUTPUT_FILE)

Repository already exists. Pulling latest changes...
/content/pml_final_project
Already up to date.
Root directory: /content/pml_final_project
Raw XLS file: /content/pml_final_project/data/raw/Area_total_de_vinha_PT_2025.xls
Raw XLSX file: /content/pml_final_project/data/raw/Area_total_de_vinha_PT_2025.xlsx
Output file: /content/pml_final_project/data/processed/vineyard_area_by_region_clean.csv


## 3. Prepare and Read the Raw File

The workbook is an `.xls` file. This notebook prefers a converted `.xlsx` file if one exists, but can also read the `.xls` directly with `xlrd`.

In [3]:
def try_convert_xls_to_xlsx(raw_xls: Path, raw_xlsx: Path) -> Path | None:
    if raw_xlsx.exists():
        print(f"Using existing XLSX file: {raw_xlsx}")
        return raw_xlsx

    if not raw_xls.exists():
        raise FileNotFoundError(
            f"Neither {raw_xlsx.name} nor {raw_xls.name} was found in data/raw."
        )

    print("Converted XLSX file was not found. Trying LibreOffice conversion...")

    try:
        if shutil.which("libreoffice") is None:
            subprocess.run(["apt-get", "update", "-qq"], check=True)
            subprocess.run(["apt-get", "install", "-y", "-qq", "libreoffice"], check=True)

        result = subprocess.run(
            [
                "libreoffice",
                "--headless",
                "--convert-to",
                "xlsx",
                "--outdir",
                str(raw_xls.parent),
                str(raw_xls),
            ],
            capture_output=True,
            text=True,
        )

        if result.returncode == 0 and raw_xlsx.exists():
            print(f"Created XLSX file: {raw_xlsx}")
            return raw_xlsx

        print("LibreOffice conversion did not create an XLSX file. Falling back to direct XLS read.")
        print(result.stderr)
        return None
    except Exception as error:
        print("LibreOffice conversion failed. Falling back to direct XLS read.")
        print(error)
        return None


EXCEL_FILE = try_convert_xls_to_xlsx(RAW_XLS, RAW_XLSX)

if EXCEL_FILE is not None:
    raw_df = pd.read_excel(EXCEL_FILE, sheet_name=0, header=None, dtype=object, engine="openpyxl")
else:
    raw_df = pd.read_excel(RAW_XLS, sheet_name=0, header=None, dtype=object, engine="xlrd")

print("Raw shape:", raw_df.shape)
display(raw_df.head(12))

Converted XLSX file was not found. Trying LibreOffice conversion...
Created XLSX file: /content/pml_final_project/data/raw/Area_total_de_vinha_PT_2025.xlsx
Raw shape: (44, 29)


,0,1,2,3,4,5,6,7,8,9,...,19,20,21,22,23,24,25,26,27,28
0,Evolução da Área Total de Vinha - Portugal (ha),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Região\nVitivinícola,1989,1999,2000-09-01 00:00:00,2001-09-01 00:00:00,2002-09-01 00:00:00,2003-09-01 00:00:00,2004-09-01 00:00:00,2005-09-01 00:00:00,2006-09-01 00:00:00,...,2016-07-31 00:00:00,2017-07-31 00:00:00,2018-07-31 00:00:00,2019-07-31 00:00:00,2020-07-31 00:00:00,2021-07-31 00:00:00,2022-07-31 00:00:00,2023-07-31 00:00:00,2024-07-31 00:00:00,2025-07-31 00:00:00
3,Verdes,38349,39638,34035,34255,34018.398,33675.821034,32881.085071,32658.573648,32235.785029,...,21019.73,21306.89,21972.97,23999.45,24240.29,24371.07,24371,23147,23234.4915,23272.15
4,Trás-os-Montes / Douro,76695,72746,67638,68404,69060.622,68713.333292,68455.072778,69107.214476,69061.722106,...,57147.06,56533.77,56094.47,56115.17,55774.63,54881.67,53167,53643,48463.312958,48450.35
5,Trás-os Montes,---,---,---,---,---,---,---,---,---,...,14380.78,14510.35,13538.66,12252.48,11613.03,10701.41,9701,9419,4565.433858,4520.63
6,Douro,---,---,---,---,---,---,---,---,---,...,42766.28,42023.42,42555.81,43862.69,44161.6,44180.26,43466,44224,43897.8791,43929.72
7,Beiras,56637,53286,57200,57608,57404.9313,57487.188054,56910.382282,57410.193341,57485.229542,...,47940.16,47653.22,46400.65,44089.61,42820.6,42756.65,43134,36245,32893.8477,32448.08
8,Távora-Varosa,---,---,---,---,---,---,---,---,---,...,2520.43,2160.85,2183.89,2345.76,2271.93,2214.88,2274,2169,2107.8999,2072.41
9,Bairrada,---,---,---,---,---,---,---,---,---,...,15086.14,15134.45,14630.3,13692.82,13314.01,13258.64,13391,9440,9304.658,9180.68


## 4. Define Cleaning Helpers

In [4]:
REGION_MAP = {
    "Verdes": "Verdes",
    "Trás-os Montes": "T. Montes",
    "Tras-os Montes": "T. Montes",
    "Douro": "Douro",
    "Bairrada": "Bairrada",
    "Dão": "Dão",
    "Dao": "Dão",
    "Beira Interior": "Beira Interior",
    "Távora-Varosa": "Távora-Varosa",
    "Tavora-Varosa": "Távora-Varosa",
    "Tejo": "Tejo",
    "Lisboa": "Lisboa",
    "Península de Setúbal": "P. Setúbal",
    "Peninsula de Setubal": "P. Setúbal",
    "Alentejo": "Alentejo",
    "Algarve": "Algarve",
    "Madeira": "Madeira",
    "Açores": "Açores",
    "Acores": "Açores",
}

REGIONS_TO_KEEP = [
    "Verdes",
    "T. Montes",
    "Douro",
    "Bairrada",
    "Dão",
    "Beira Interior",
    "Távora-Varosa",
    "Tejo",
    "Lisboa",
    "P. Setúbal",
    "Alentejo",
    "Algarve",
    "Madeira",
    "Açores",
]


def normalize_text(value):
    if pd.isna(value):
        return ""
    text = str(value).replace("\xa0", " ").strip()
    text = re.sub(r"\s+", " ", text)
    return text


def normalize_for_matching(value):
    text = normalize_text(value)
    replacements = {
        "ã": "a",
        "á": "a",
        "à": "a",
        "â": "a",
        "ç": "c",
        "é": "e",
        "ê": "e",
        "í": "i",
        "ó": "o",
        "ô": "o",
        "õ": "o",
        "ú": "u",
        "Á": "A",
        "Ã": "A",
        "Ç": "C",
        "É": "E",
        "Í": "I",
        "Ó": "O",
        "Ú": "U",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


REGION_MATCH = {normalize_for_matching(raw): clean for raw, clean in REGION_MAP.items()}


def parse_year(value):
    if pd.isna(value):
        return None

    if isinstance(value, (pd.Timestamp, datetime)):
        return int(value.year)

    text = normalize_text(value)

    match = re.search(r"(19\d{2}|20\d{2})", text)
    if match:
        return int(match.group(1))

    return None


def clean_number(value):
    if isinstance(value, (int, float, np.integer, np.floating)) and not isinstance(value, bool):
        return float(value)

    text = normalize_text(value)
    if text == "" or text.lower() in {"nan", "none", "---"}:
        return np.nan

    text = re.sub(r"[^0-9,.-]", "", text)
    if text in {"", "-", ".", ","}:
        return np.nan

    has_comma = "," in text
    has_dot = "." in text

    if has_comma and has_dot:
        if text.rfind(",") > text.rfind("."):
            text = text.replace(".", "").replace(",", ".")
        else:
            text = text.replace(",", "")
    elif has_comma:
        text = text.replace(",", ".")
    elif has_dot and re.fullmatch(r"-?\d{1,3}(\.\d{3})+", text):
        text = text.replace(".", "")

    try:
        return float(text)
    except ValueError:
        return np.nan

## 5. Extract Vineyard Area Table

In [5]:
header_row = None

for row_idx in range(len(raw_df)):
    row_values = [parse_year(value) for value in raw_df.iloc[row_idx].tolist()]
    year_count = sum(year is not None for year in row_values)
    first_cell = normalize_for_matching(raw_df.iat[row_idx, 0]).lower()

    if "regiao" in first_cell and year_count >= 5:
        header_row = row_idx
        break

if header_row is None:
    raise ValueError("Could not find the vineyard area header row.")

print("Header row:", header_row)

year_columns = {
    col_idx: parse_year(value)
    for col_idx, value in enumerate(raw_df.iloc[header_row].tolist())
    if parse_year(value) is not None
}

print("Years found:", sorted(set(year_columns.values())))

records = []

for row_idx in range(header_row + 1, len(raw_df)):
    raw_region = normalize_text(raw_df.iat[row_idx, 0])
    region_key = normalize_for_matching(raw_region)

    if region_key.lower() in {"total geral", "total - continente", "total - regioes autonomas"}:
        continue

    if region_key not in REGION_MATCH:
        continue

    region = REGION_MATCH[region_key]

    for col_idx, year_start in year_columns.items():
        vineyard_area_ha = clean_number(raw_df.iat[row_idx, col_idx])

        if pd.isna(vineyard_area_ha):
            continue

        records.append(
            {
                "region": region,
                "year_start": int(year_start),
                "vineyard_area_ha": vineyard_area_ha,
            }
        )

vineyard_area = pd.DataFrame(records)
vineyard_area = vineyard_area[vineyard_area["region"].isin(REGIONS_TO_KEEP)]
vineyard_area = vineyard_area.drop_duplicates(subset=["region", "year_start"], keep="first")
vineyard_area = vineyard_area.sort_values(["year_start", "region"]).reset_index(drop=True)
vineyard_area["vineyard_area_ha"] = vineyard_area["vineyard_area_ha"].round(2)

display(vineyard_area.head(20))
print("Clean dataset shape:", vineyard_area.shape)

Header row: 2
Years found: [1989, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


,region,year_start,vineyard_area_ha
0,Alentejo,1989,11510.00
1,Algarve,1989,2750.00
2,Açores,1989,2489.00
3,Lisboa,1989,46046.00
4,Madeira,1989,1803.00
5,P. Setúbal,1989,11396.00
6,Tejo,1989,28124.00
7,Verdes,1989,38349.00
8,Alentejo,1999,13457.00
9,Algarve,1999,1933.00


Clean dataset shape: (288, 3)


## 6. Quality Checks

In [6]:
print("Dataset shape:", vineyard_area.shape)

print("\nColumns:")
print(vineyard_area.columns.tolist())

print("\nYears:")
print(vineyard_area["year_start"].min(), "to", vineyard_area["year_start"].max())
print(sorted(vineyard_area["year_start"].unique()))

print("\nRegions:")
print(sorted(vineyard_area["region"].unique()))

print("\nDuplicated region-year rows:")
print(vineyard_area.duplicated(subset=["region", "year_start"]).sum())

print("\nMissing values:")
display(vineyard_area.isna().sum().to_frame("missing_count"))

print("\nSummary statistics for vineyard_area_ha:")
display(vineyard_area["vineyard_area_ha"].describe())

print("\nRows per region:")
display(vineyard_area.groupby("region")["year_start"].agg(["min", "max", "count"]).reset_index())

Dataset shape: (288, 3)

Columns:
['region', 'year_start', 'vineyard_area_ha']

Years:
1989 to 2025
[np.int64(1989), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Regions:
['Alentejo', 'Algarve', 'Açores', 'Bairrada', 'Beira Interior', 'Douro', 'Dão', 'Lisboa', 'Madeira', 'P. Setúbal', 'T. Montes', 'Tejo', 'Távora-Varosa', 'Verdes']

Duplicated region-year rows:
0

Missing values:


,missing_count
region,0
year_start,0
vineyard_area_ha,0



Summary statistics for vineyard_area_ha:


,vineyard_area_ha
count,288.000000
mean,14054.336563
std,11818.533418
min,645.880000
25%,1985.497500
50%,12634.000000
75%,23103.500000
max,46046.000000



Rows per region:


,region,min,max,count
0,Alentejo,1989,2025,28
1,Algarve,1989,2025,28
2,Açores,1989,2025,28
3,Bairrada,2016,2025,10
4,Beira Interior,2016,2025,10
5,Douro,2014,2025,12
6,Dão,2016,2025,10
7,Lisboa,1989,2025,28
8,Madeira,1989,2025,28
9,P. Setúbal,1989,2025,28


## 7. Save Processed CSV

In [7]:
vineyard_area.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("Saved processed dataset to:")
print(OUTPUT_FILE)

Saved processed dataset to:
/content/pml_final_project/data/processed/vineyard_area_by_region_clean.csv


## 8. Optional Merge Check with Wine Production

This check is only to confirm that vineyard area can be used later as an explanatory variable for wine production models.

In [8]:
PRODUCTION_FILE = ROOT_DIR / "data" / "processed" / "wine_production_by_region_clean.csv"

if PRODUCTION_FILE.exists():
    production = pd.read_csv(PRODUCTION_FILE)
    production["year_start"] = pd.to_numeric(production["year_start"], errors="coerce").astype("Int64")

    merged = production.merge(
        vineyard_area,
        on=["region", "year_start"],
        how="left",
    )

    print("Production shape:", production.shape)
    print("Merged shape:", merged.shape)
    print("Rows with missing vineyard_area_ha:", merged["vineyard_area_ha"].isna().sum())
    display(merged.head(20))
else:
    print("Production file not found yet. Run ivv_wine_production.ipynb first to test the merge.")

Production shape: (238, 8)
Merged shape: (238, 9)
Rows with missing vineyard_area_ha: 38


,region,campaign_year,year_start,total_production_hl,dop_production_hl,igp_production_hl,year_variety_production_hl,non_certified_production_hl,vineyard_area_ha
0,Alentejo,2009/10,2009,810338,356783,449833,995,2728,23490.00
1,Algarve,2009/10,2009,23650,4680,11797,0,7173,1983.00
2,Açores,2009/10,2009,13754,2771,2832,6,8145,1700.00
3,Bairrada,2009/10,2009,246705,69459,64138,1068,112041,NaN
4,Beira Interior,2009/10,2009,192084,50158,20570,14,121342,NaN
5,Douro,2009/10,2009,1351949,1186358,26986,0,138605,NaN
6,Dão,2009/10,2009,297483,221529,23022,0,52932,NaN
7,Lisboa,2009/10,2009,962323,55210,323594,0,583519,24799.00
8,Madeira,2009/10,2009,45449,39285,223,0,5940,1458.66
9,P. Setúbal,2009/10,2009,379371,108271,178274,507,92319,9210.00


## 9. Download CSV

In [9]:
from google.colab import files

files.download(str(OUTPUT_FILE))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>